In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
print("[STEP 1] Bronze -> Silver job started. Parameters received:")
print(f"    landing_timestamp : '{landing_timestamp}'")
print(f"    table_name        : '{table_name}'")
print(f"    merge_keys        : '{merge_keys}'")
print(f"    load_type         : '{load_type}'")
print("=" * 80)

In [0]:
source_table = f"hcp_databricks_workspace.bronze.{table_name}"
filter_condition = f"landing_timestamp = '{landing_timestamp}'"

print("[STEP 2] Reading data from Bronze layer")
print(f"    Source table : {source_table}")
print(f"    Filter       : {filter_condition}")
print("    Equivalent SQL:")
print(f"        SELECT * FROM {source_table} WHERE {filter_condition}")

bronze_df = spark.read.table(source_table).filter(filter_condition)

bronze_count = bronze_df.count()
print(f"    Rows read from Bronze : {bronze_count}")
print(f"    Columns ({len(bronze_df.columns)}) : {bronze_df.columns}")
if bronze_count == 0:
    print("    WARNING: No rows found in Bronze for the given landing_timestamp.")
print("-" * 80)

In [0]:
from pyspark.sql.functions import current_timestamp

print("[STEP 3] Adding audit columns to the data")
print("    insert_timestamp = current_timestamp()")
print("    update_timestamp = current_timestamp()")

updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())

print(f"    Final columns ({len(updated_df.columns)}) : {updated_df.columns}")
print("-" * 80)

In [0]:
from delta.tables import DeltaTable

target_table = f"hcp_databricks_workspace.silver.{table_name}"
load_type_normalized = load_type.strip().upper()
cols_sql = ", ".join([f"`{c}`" for c in updated_df.columns])

print("[STEP 4] Writing data to Silver layer")
print(f"    Target table : {target_table}")
print(f"    Load type    : {load_type_normalized}")

if load_type_normalized == "FULL":
    # Full load: overwrite target table
    print("    Mode: FULL -> target table will be overwritten")
    print("    Equivalent SQL:")
    print(f"        CREATE OR REPLACE TABLE {target_table} AS")
    print(f"        SELECT {cols_sql} FROM <bronze_with_audit_columns>")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print(f"    FULL load completed for {target_table}")

elif load_type_normalized == "APPEND":
    # Append load: add new rows to target table
    print("    Mode: APPEND -> new rows will be added to target table")
    print("    Equivalent SQL:")
    print(f"        INSERT INTO {target_table} ({cols_sql})")
    print(f"        SELECT {cols_sql} FROM <bronze_with_audit_columns>")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"    APPEND load completed for {target_table}")

elif load_type_normalized == "MERGE":
    # Merge load: update existing rows and insert new rows based on merge_keys
    print("    Mode: MERGE -> update matching rows, insert new rows")
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    print(f"    Parsed merge keys : {keys}")
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type = 'MERGE'")

    missing_keys = [k for k in keys if k not in updated_df.columns]
    if missing_keys:
        raise ValueError(f"Merge keys not found in source data: {missing_keys}")

    # Deduplicate source on merge keys to avoid multiple-match MERGE errors
    source_df = updated_df.dropDuplicates(keys)
    print(f"    Source deduplicated on {keys}")

    table_exists = spark.catalog.tableExists(target_table)
    print(f"    Target table exists : {table_exists}")

    if not table_exists:
        # First run: create the target table
        print("    First run -> creating target table from source data")
        print("    Equivalent SQL:")
        print(f"        CREATE TABLE {target_table} AS")
        print(f"        SELECT {cols_sql} FROM <deduplicated_source>")
        source_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"    Target table {target_table} created")
    else:
        merge_condition = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        # On update, keep the original insert_timestamp and refresh everything else
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}
        insert_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        print(f"    Merge condition : {merge_condition}")
        print("    Columns updated on match (insert_timestamp excluded):")
        print(f"        {list(update_set.keys())}")

        update_sql = ",\n            ".join([f"t.{k} = {v}" for k, v in update_set.items()])
        insert_cols_sql = ", ".join(insert_set.keys())
        insert_vals_sql = ", ".join(insert_set.values())
        merge_sql = (
            f"        MERGE INTO {target_table} AS t\n"
            f"        USING <deduplicated_source> AS s\n"
            f"        ON {merge_condition}\n"
            f"        WHEN MATCHED THEN UPDATE SET\n"
            f"            {update_sql}\n"
            f"        WHEN NOT MATCHED THEN INSERT ({insert_cols_sql})\n"
            f"            VALUES ({insert_vals_sql})"
        )
        print("    Equivalent SQL:")
        print(merge_sql)

        (
            DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_set)
            .execute()
        )

        # Report MERGE metrics from the Delta history
        last_op = DeltaTable.forName(spark, target_table).history(1).select("operation", "operationMetrics").collect()[0]
        m = last_op["operationMetrics"] or {}
        print(f"    MERGE completed. Operation: {last_op['operation']}")
        print(f"        Rows updated  : {m.get('numTargetRowsUpdated', 'n/a')}")
        print(f"        Rows inserted : {m.get('numTargetRowsInserted', 'n/a')}")
        print(f"        Source rows   : {m.get('numSourceRows', 'n/a')}")

else:
    print(f"    ERROR: Unsupported load_type '{load_type}'")
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

print("-" * 80)

In [0]:
record_count = updated_df.count()
print("[STEP 5] Job finished")
print(f"    Records processed : {record_count}")
print(f"    Returning record count '{record_count}' to the caller via dbutils.notebook.exit")
print("=" * 80)
dbutils.notebook.exit(str(record_count))